# Baselione training

Loads only the training set, adds weekday and month. adds zero

In [ ]:
# load data

import polars as pl
from pathlib import Path
import pandas as pd
import pyarrow

PROCESSED_FOLDER = Path("./../data/processed/")

data = pl.read_parquet(PROCESSED_FOLDER / "train.parquet").to_pandas()
data.head()

,date,store_nbr,item_nbr,unit_sales,onpromotion
0,2013-01-01,25,103665,7.0,None
1,2013-01-01,25,105574,1.0,None
2,2013-01-01,25,105575,2.0,None
3,2013-01-01,25,108079,1.0,None
4,2013-01-01,25,108701,1.0,None


In [3]:
# data cleaning

# remove dates before august 2015
# data = data[data["date"] >= "2015-08-16"]

# adds weekday and month feature
data["date"] = pd.to_datetime(data["date"])
data["weekday_num"] = data["date"].dt.day_of_week
data["month_num"] = data["date"].dt.month

# clips unit sales to 1
data["unit_sales"] = data["unit_sales"].clip(lower=0)

data.head()


,date,store_nbr,item_nbr,unit_sales,onpromotion,weekday_num,month_num
0,2013-01-01,25,103665,7.0,None,1,1
1,2013-01-01,25,105574,1.0,None,1,1
2,2013-01-01,25,105575,2.0,None,1,1
3,2013-01-01,25,108079,1.0,None,1,1
4,2013-01-01,25,108701,1.0,None,1,1


In [4]:
# sets everything with none on false promotion

data["onpromotion"] = data["onpromotion"].fillna(False)

data.head()


,date,store_nbr,item_nbr,unit_sales,onpromotion,weekday_num,month_num
0,2013-01-01,25,103665,7.0,False,1,1
1,2013-01-01,25,105574,1.0,False,1,1
2,2013-01-01,25,105575,2.0,False,1,1
3,2013-01-01,25,108079,1.0,False,1,1
4,2013-01-01,25,108701,1.0,False,1,1


In [ ]:
# adds 0 sales for missing item sales

sales = data.groupby(["date", "store_nbr", "item_nbr"])["unit_sales"].sum()

all_dates = sales.index.get_level_values("date").unique()
all_items = sales.index.get_level_values("item_nbr").unique()
all_shops = sales.index.get_level_values("store_nbr").unique()

full_index = pd.MultiIndex.from_product([all_dates, all_shops, all_items], names=["date", "store_nbr", "item_nbr"])

data_zeros = sales.reindex(full_index, fill_value=0).reset_index()

data_zeros.head(10)


date        store_nbr  item_nbr
2013-01-01  25         103665      7.0
                       105574      1.0
                       105575      2.0
                       108079      1.0
                       108701      1.0
Name: unit_sales, dtype: float32

In [6]:
data_merged = data_zeros.merge(data.drop(columns="unit_sales"), on=["date", "item_nbr", "store_nbr"], how="left")
data_merged.head()


,date,store_nbr,item_nbr,unit_sales,onpromotion,weekday_num,month_num
0,2013-01-01,25,103665,7.0,False,1.0,1.0
1,2013-01-01,25,105574,1.0,False,1.0,1.0
2,2013-01-01,25,105575,2.0,False,1.0,1.0
3,2013-01-01,25,108079,1.0,False,1.0,1.0
4,2013-01-01,25,108701,1.0,False,1.0,1.0


In [ ]:
# train model

df = data_merged

import numpy as np
from sklearn.preprocessing import TargetEncoder
from sklearn.ensemble import HistGradientBoostingRegressor

df["onpromotion"] = df["onpromotion"].fillna(False).astype(int)
df["target"] = np.log1p(df["unit_sales"])
df["store_nbr"] = df["store_nbr"].astype("category")

# take randon training and validation data
train = df[df["date"] < "2016-08-16"].copy().sample(n=2_000_000, random_state=0) 
valid = df[df["date"] >= "2016-08-16"].copy().sample(n=400_000, random_state=0) 

print(f"Training set: {train.shape[0]} rows.")
print(f"Test set: {valid.shape[0]} rows.")

# item_nbr: too many categories for native support -> target encoding
te = TargetEncoder()
train["item_te"] = te.fit_transform(train[["item_nbr"]], train["target"]).ravel()
valid["item_te"] = te.transform(valid[["item_nbr"]]).ravel()

print("transformed item_nb to TE.")

features = ["store_nbr", "item_te", "onpromotion", "weekday_num", "month_num"]
model = HistGradientBoostingRegressor(
    categorical_features="from_dtype",   # picks up store_nbr as a category
    max_iter=1000, learning_rate=0.05, max_leaf_nodes=127,
    early_stopping=True,
)
model.fit(train[features], train["target"])
pred = np.expm1(model.predict(valid[features]))


Training set: 2000000 rows.
Test set: 400000 rows.
transformed item_nb to TE.


## Scoring function

$$ 
NWRMSLE = \sqrt{ \frac{\sum_{i=1}^n w_i \left( \ln(\hat{y}i + 1) - \ln(y_i +1)  \right)^2  }{\sum{i=1}^n w_i}} 
$$

where for row i, ^yi is the predicted unit_sales of an item and 
yi is the actual unit_sales; n is the total number of rows in the test set.

The weights, 
, can be found in the items.csv file (see the Data page). Perishable items are given a weight of 1.25 where all other items are given a weight of 1.00.

In [8]:
# score model

from sklearn.metrics import root_mean_squared_log_error

error = root_mean_squared_log_error(pred, valid["unit_sales"])
print(f"Error for random sample set is: {error}")


Error for random sample set is: 0.5395936539984797


In [29]:
# test submission data

EXTRACTED_FOLDER = Path("./../data/raw") / "extracted"

test_data = pd.read_csv(EXTRACTED_FOLDER / "test.csv")

test_data["onpromotion"] = test_data["onpromotion"].fillna(False).astype(int)
test_data["store_nbr"] = test_data["store_nbr"].astype("category")
test_data["item_te"] = te.transform(test_data[["item_nbr"]]).ravel()

test_data["date"] = pd.to_datetime(test_data["date"])
test_data["weekday_num"] = test_data["date"].dt.day_of_week
test_data["month_num"] = test_data["date"].dt.month

test_pred = np.expm1(model.predict(test_data[features]))


In [ ]:
# create submission file

import os

RESULT_FOLDER = Path("./../data/results") 

submission_data = pd.DataFrame({
    "id": test_data["id"].to_numpy(),
    "unit_sales": np.asarray(test_pred, dtype=float).ravel(),
})

os.makedirs(RESULT_FOLDER, exist_ok=True)
submission_data.to_csv(RESULT_FOLDER / "baseline_result_v1.csv", index=False)

submission_data.head()

OSError: Cannot save file into a non-existent directory: '../data/results'